## Covolution Layer

In [7]:
import torch
from torch import nn
from d2l import torch as d2l

In [8]:
def corr2d(X, K):
    """compute 2d cross-correlatoin"""
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i+h, j:j+w] * K).sum()
    return Y

In [9]:
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
corr2d(X, K)

tensor([[19., 25.],
        [37., 43.]])

## Object Edge Detection

In [12]:
X = torch.ones((6,8))
X[:,2:6] = 0
X

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

In [14]:
# Convolution kernal
K = torch.tensor([[1.0,-1.0]])

In [16]:
Y = corr2d(X,K)
Y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

## Learning Convolution Kernel

In [31]:
# Construct a 2D convolutional layer with 1 output channel
# and a convolution kernel of shape (1, 2)
conv2d = nn.Conv2d(1, 1, kernel_size=(1,2), bias=False)

# This 2D convolutional layer uses 4D input and output formats: 
# (batch_size, channels, height, width),
# where both the batch size and the number of channels are 1.
X = X.reshape((1,1,6,8))
Y = Y.reshape((1,1,6,7))
lr = 3e-2  # learning rate

for i in range(10):
    Y_hat = conv2d(X)
    loss = (Y_hat - Y) ** 2
    conv2d.zero_grad()
    loss.sum().backward()
    # Iterate over the convolution kernel
    with torch.no_grad():
        conv2d.weight -= lr * conv2d.weight.grad
    if (i + 1) % 2 == 0:
        print(f'epoch {i+1}, loss {loss.sum():.3f}')

epoch 2, loss 5.859
epoch 4, loss 1.195
epoch 6, loss 0.287
epoch 8, loss 0.084
epoch 10, loss 0.029


In [32]:
conv2d.weight.data.reshape((1, 2))

tensor([[ 0.9722, -1.0046]])

## Padding and Stride

### Padding

In [36]:
def comp_conv2d(conv2d, X):
    # (1，1) represents batch szie and channels
    X = X.reshape((1,1) + X.shape)
    Y = conv2d(X)
    return Y.reshape(Y.shape[2:])

# each side was filled in 1 row and 1 column, so 2 rows and 2 columns were added to the X
conv2d = nn.Conv2d(1, 1, kernel_size=3, padding=1)
X = torch.rand((8, 8))
comp_conv2d(conv2d, X).shape

torch.Size([8, 8])

In [37]:
conv2d = nn.Conv2d(1, 1, kernel_size=(5, 3), padding=(2,1))
comp_conv2d(conv2d, X).shape

torch.Size([8, 8])

### Stride

In [38]:
conv2d = nn.Conv2d(1, 1, kernel_size=3, padding=1, stride=2)
comp_conv2d(conv2d, X).shape

torch.Size([4, 4])

In [39]:
conv2d = nn.Conv2d(1, 1, kernel_size=(3,5), padding=(0,1), stride=(3,4))
comp_conv2d(conv2d, X).shape

torch.Size([2, 2])

# Channels

### Multi-Input Channels

In [43]:
def corr2d_multi_in(X, K):
    # iterate the dimension 0 of X and K, then add them together
    return sum(corr2d(x, k) for x, k in zip(X, K))

In [44]:
X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
               [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])

corr2d_multi_in(X, K)

tensor([[ 56.,  72.],
        [104., 120.]])

### Multi-Output Channels

In [45]:
def corr2d_multi_in_out(X, K):
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)

In [46]:
K = torch.stack((K, K + 1, K + 2), 0)
K.shape

torch.Size([3, 2, 2, 2])

In [47]:
corr2d_multi_in_out(X, K)

tensor([[[ 56.,  72.],
         [104., 120.]],

        [[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]]])

### 1 X 1 Convolution Kernel Layer

In [48]:
def corr2d_multi_in_out_1x1(X, K):
    c_i, h, w = X.shape
    c_o = K.shape[0]
    X = X.reshape((c_i, h*w))
    K = K.reshape((c_o, c_i))
    # Matrix multiplication in the fully connected layer
    Y = torch.matmul(K,X)
    return Y.reshape((c_o, h, w))

In [49]:
X = torch.normal(0, 1, (3, 3, 3))
K = torch.normal(0, 1, (2, 3, 1, 1))

In [50]:
Y1 = corr2d_multi_in_out_1x1(X, K)
Y2 = corr2d_multi_in_out(X, K)
assert float(torch.abs(Y1 - Y2).sum()) < 1e-6

## Pooling

In [ ]:
def pool2d(X, pool_size, mode='max'):
    p_h, p_w = pool_size
    Y = torch.zeros((X.shape[0] - ))